In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [ ]:
columns = ["SepalLength", "SepalWidth", "PetalLength", "PetalWidth", "Species"]

df = pd.read_csv(
    "../data/iris.data",
    sep=r"\s+",
    skiprows=2,
    names=columns
)

df.head()

In [ ]:
df.info()
df["Species"].value_counts()

In [ ]:
X = df[["SepalLength", "SepalWidth", "PetalLength", "PetalWidth"]]
y_true = df["Species"]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

clusters = kmeans.fit_predict(X_scaled)

df["Cluster"] = clusters

df.head()

In [ ]:
pd.crosstab(df["Species"], df["Cluster"])

In [ ]:
cluster_to_species = {}

for cluster in sorted(df["Cluster"].unique()):
    most_common_species = df[df["Cluster"] == cluster]["Species"].mode()[0]
    cluster_to_species[cluster] = most_common_species

cluster_to_species

df["PredictedSpecies"] = df["Cluster"].map(cluster_to_species)

df.head()

In [ ]:
labels = ["Se", "Ve", "Vi"]

cm = confusion_matrix(y_true, df["PredictedSpecies"], labels=labels)

confusion_df = pd.DataFrame(
    cm,
    index=["Actual Se", "Actual Ve", "Actual Vi"],
    columns=["Predicted Se", "Predicted Ve", "Predicted Vi"]
)

confusion_df

In [ ]:
results = []

for i, label in enumerate(labels):
    TP = cm[i, i]
    FN = cm[i, :].sum() - TP
    
    results.append({
        "Class": label,
        "True Positive": TP,
        "False Negative": FN
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
accuracy = accuracy_score(y_true, df["PredictedSpecies"])

print("Accuracy:", accuracy)

print(classification_report(y_true, df["PredictedSpecies"], labels=labels))